In [6]:
import numpy as np
import pandas as pd
import warnings
import numpy as np

from itertools import combinations, product
from math import comb
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef
)

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [3]:
df = pd.read_csv(r"../../../data/processed/data_vif.csv")
target = "Risk_Label"

n_total = len(df)
n_train = int(n_total * 0.5)
n_valid = int(n_total * 0.3)
n_test = n_total - n_train - n_valid

#Date 컬럼 제거
df.drop("Date", axis=1, inplace=True)

# Risk_Label 매핑 (High Risk=1, Low Risk=0)
df["Risk_Label"] = df["Risk_Label"].map({"Low Risk": 0, "High Risk": 1})

#train/valid/test 분할 5/3/2
train_df = df.iloc[:n_train].reset_index(drop=True)
valid_df = df.iloc[n_train : n_train + n_valid].reset_index(drop=True)
test_df = df.iloc[n_train + n_valid :].reset_index(drop=True)

print(f"total rows: {n_total}, train: {len(train_df)}, valid: {len(valid_df)}, test: {len(test_df)}")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

total rows: 4108, train: 2054, valid: 1232, test: 822
train shape: (2054, 28) (2054,)
valid shape: (1232, 28) (1232,)
test shape: (822, 28) (822,)


In [4]:
scaler = MinMaxScaler().set_output(transform="pandas")

# train에만 fit하고 valid/test에는 같은 scaler를 적용
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

print('train/valid/test:', len(X_train), len(X_valid), len(X_test))
print('y_train class:', y_train.value_counts().to_dict())
print('y_valid class:', y_valid.value_counts().to_dict())
print('y_test class:', y_test.value_counts().to_dict())

train/valid/test: 2054 1232 822
y_train class: {0: 1859, 1: 195}
y_valid class: {0: 1070, 1: 162}
y_test class: {0: 730, 1: 92}


In [ ]:
# =========================
# Stepwise Selection Param Search Space
# =========================

param_space = {
    "C": [0.1, 0.5, 1.0, 3.0, 5.0, 10.0],
    "class_weight": ["balanced"],
    "solver": ["liblinear"],
    "threshold": [0.1, 0.2, 0.3, 0.35, 0.40, 0.45, 0.50, 0.55]
}


def make_param_list(param_space):
    keys = list(param_space.keys())
    values = list(param_space.values())

    param_list = []

    for comb in product(*values):
        param_list.append(dict(zip(keys, comb)))

    return param_list


param_list = make_param_list(param_space)

총 파라미터 조합 수: 48


[{'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.1},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.2},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.3},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.35},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.4}]

## StepWise

In [11]:
def get_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    gmean = np.sqrt(recall * spec)
    metrics = {
        "gmean": gmean,
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision,
        "recall": recall,
        "spec": spec,
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    return metrics
def fit_eval_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    features,
    param_list,
    random_state=1,
    max_iter=2000
):
    """
    주어진 변수 조합 features에 대해
    param_list 전체 탐색 후 validation gmean이 가장 좋은 모델 반환
    """

    best_record = None

    for params in param_list:
        model = LogisticRegression(
            penalty="l2",
            C=params["C"],
            class_weight=params["class_weight"],
            solver=params["solver"],
            max_iter=max_iter,
            random_state=random_state
        )

        model.fit(X_train[features], y_train)

        y_proba = model.predict_proba(X_valid[features])[:, 1]

        metrics = get_metrics(
            y_true=y_valid,
            y_proba=y_proba,
            threshold=params["threshold"]
        )

        record = {
            "features": features.copy(),
            "params": params,
            "metrics": metrics,
            "model": model
        }

        if best_record is None:
            best_record = record
        elif metrics["gmean"] > best_record["metrics"]["gmean"]:
            best_record = record

    return best_record


def stepwise_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    param_list,
    min_delta=0.001,
    random_state=1,
    max_iter=2000,
    verbose=True
):
    """
    Logistic Regression 기반 Stepwise Selection.

    방식:
        1. 현재 변수 집합에 후보 변수를 하나씩 추가해봄
        2. validation gmean이 가장 좋아지는 변수를 추가
        3. 추가 후, 현재 선택된 변수 중 하나씩 제거해봄
        4. 제거해서 gmean이 더 좋아지면 제거
        5. 더 이상 추가/제거 개선이 없으면 종료

    기준:
        validation gmean
    """

    selected_features = []
    remaining_features = list(X_train.columns)

    best_score = -np.inf
    best_model = None
    best_params = None
    best_metrics = None

    step_history = []
    candidate_history = []

    step = 1

    while True:
        changed = False

        # =========================
        # Forward Step
        # =========================

        forward_best = None

        if verbose:
            print(f"\n[STEP {step} - FORWARD]")
            print(f"현재 선택 변수 수: {len(selected_features)}")
            print(f"추가 후보 변수 수: {len(remaining_features)}")

        for add_feature in remaining_features:
            trial_features = selected_features + [add_feature]

            record = fit_eval_lr(
                X_train=X_train,
                y_train=y_train,
                X_valid=X_valid,
                y_valid=y_valid,
                features=trial_features,
                param_list=param_list,
                random_state=random_state,
                max_iter=max_iter
            )

            candidate_history.append({
                "step": step,
                "action": "add",
                "feature": add_feature,
                "n_features": len(trial_features),
                **record["params"],
                **record["metrics"]
            })

            if forward_best is None:
                forward_best = {
                    "action": "add",
                    "feature": add_feature,
                    **record
                }
            elif record["metrics"]["gmean"] > forward_best["metrics"]["gmean"]:
                forward_best = {
                    "action": "add",
                    "feature": add_feature,
                    **record
                }

        if forward_best is not None:
            improvement = forward_best["metrics"]["gmean"] - best_score

            if improvement > min_delta:
                selected_features = forward_best["features"]
                remaining_features.remove(forward_best["feature"])

                best_score = forward_best["metrics"]["gmean"]
                best_model = forward_best["model"]
                best_params = forward_best["params"]
                best_metrics = forward_best["metrics"]

                step_history.append({
                    "step": step,
                    "action": "add",
                    "feature": forward_best["feature"],
                    "n_features": len(selected_features),
                    "improvement": improvement if np.isfinite(improvement) else np.nan,
                    **best_params,
                    **best_metrics
                })

                changed = True

                if verbose:
                    print(
                        f"추가 변수: {forward_best['feature']} | "
                        f"gmean={best_score:.4f} | "
                        f"threshold={best_params['threshold']} | "
                        f"C={best_params['C']} | "
                        f"class_weight={best_params['class_weight']}"
                    )
            else:
                if verbose:
                    print(f"추가 개선폭 {improvement:.6f} <= min_delta {min_delta}")

        # =========================
        # Backward Step
        # =========================

        backward_improved = True

        while backward_improved and len(selected_features) > 1:
            backward_improved = False
            backward_best = None

            if verbose:
                print(f"\n[STEP {step} - BACKWARD]")
                print(f"현재 선택 변수 수: {len(selected_features)}")
                print("선택 변수 중 하나씩 제거하면서 평가 중...")

            for remove_feature in selected_features:
                trial_features = [
                    f for f in selected_features
                    if f != remove_feature
                ]

                record = fit_eval_lr(
                    X_train=X_train,
                    y_train=y_train,
                    X_valid=X_valid,
                    y_valid=y_valid,
                    features=trial_features,
                    param_list=param_list,
                    random_state=random_state,
                    max_iter=max_iter
                )

                candidate_history.append({
                    "step": step,
                    "action": "remove",
                    "feature": remove_feature,
                    "n_features": len(trial_features),
                    **record["params"],
                    **record["metrics"]
                })

                if backward_best is None:
                    backward_best = {
                        "action": "remove",
                        "feature": remove_feature,
                        **record
                    }
                elif record["metrics"]["gmean"] > backward_best["metrics"]["gmean"]:
                    backward_best = {
                        "action": "remove",
                        "feature": remove_feature,
                        **record
                    }

            improvement = backward_best["metrics"]["gmean"] - best_score

            if improvement > min_delta:
                selected_features = backward_best["features"]
                remaining_features.append(backward_best["feature"])

                best_score = backward_best["metrics"]["gmean"]
                best_model = backward_best["model"]
                best_params = backward_best["params"]
                best_metrics = backward_best["metrics"]

                step_history.append({
                    "step": step,
                    "action": "remove",
                    "feature": backward_best["feature"],
                    "n_features": len(selected_features),
                    "improvement": improvement,
                    **best_params,
                    **best_metrics
                })

                changed = True
                backward_improved = True

                if verbose:
                    print(
                        f"제거 변수: {backward_best['feature']} | "
                        f"gmean={best_score:.4f} | "
                        f"improvement={improvement:.6f} | "
                        f"threshold={best_params['threshold']} | "
                        f"C={best_params['C']} | "
                        f"class_weight={best_params['class_weight']}"
                    )
            else:
                if verbose:
                    print(f"제거 개선폭 {improvement:.6f} <= min_delta {min_delta}")

        if not changed:
            if verbose:
                print("\nStepwise Selection 종료")
            break

        if len(remaining_features) == 0:
            if verbose:
                print("\n남은 후보 변수가 없어서 종료")
            break

        step += 1

    result = {
        "selected_features": selected_features,
        "best_model": best_model,
        "best_params": best_params,
        "best_metrics": best_metrics,
        "step_history": pd.DataFrame(step_history),
        "candidate_history": pd.DataFrame(candidate_history)
    }

    return result

In [12]:
stepwise_result = stepwise_selection_lr(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    param_list=param_list,
    min_delta=0.001,
    random_state=1,
    max_iter=2000,
    verbose=True
)


print("\n" + "=" * 80)
print("최종 선택 변수")
print("=" * 80)
print(stepwise_result["selected_features"])


print("\n" + "=" * 80)
print("Best Parameters")
print("=" * 80)
print(stepwise_result["best_params"])


metric_order = [
    "gmean",
    "threshold",
    "acc",
    "f1",
    "precision",
    "recall",
    "spec",
    "balanced_acc",
    "mcc",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp"
]


print("\n" + "=" * 80)
print("Validation Metrics")
print("=" * 80)

metrics_df = pd.DataFrame([stepwise_result["best_metrics"]])[metric_order]
display(metrics_df)


print("\n" + "=" * 80)
print("Validation Confusion Matrix")
print("row=true, col=pred")
print("[[TN, FP],")
print(" [FN, TP]]")
print("=" * 80)

cm = np.array([
    [stepwise_result["best_metrics"]["tn"], stepwise_result["best_metrics"]["fp"]],
    [stepwise_result["best_metrics"]["fn"], stepwise_result["best_metrics"]["tp"]]
])

print(cm)


print("\n" + "=" * 80)
print("Step History")
print("=" * 80)

step_cols = [
    "step",
    "action",
    "feature",
    "n_features",
    "gmean",
    "improvement",
    "threshold",
    "acc",
    "f1",
    "precision",
    "recall",
    "spec",
    "balanced_acc",
    "tn",
    "fp",
    "fn",
    "tp",
    "C",
    "class_weight",
    "solver"
]

if len(stepwise_result["step_history"]) > 0:
    display(stepwise_result["step_history"][step_cols])
else:
    print("선택된 변수가 없습니다.")


# =========================
# Test set 평가
# =========================

try:
    best_model = stepwise_result["best_model"]
    selected_features = stepwise_result["selected_features"]
    best_threshold = stepwise_result["best_params"]["threshold"]

    y_test_proba = best_model.predict_proba(X_test[selected_features])[:, 1]

    test_metrics = get_metrics(
        y_true=y_test,
        y_proba=y_test_proba,
        threshold=best_threshold
    )

    print("\n" + "=" * 80)
    print("Test Metrics")
    print("=" * 80)

    test_metrics_df = pd.DataFrame([test_metrics])[metric_order]
    display(test_metrics_df)

    print("\n" + "=" * 80)
    print("Test Confusion Matrix")
    print("row=true, col=pred")
    print("[[TN, FP],")
    print(" [FN, TP]]")
    print("=" * 80)

    test_cm = np.array([
        [test_metrics["tn"], test_metrics["fp"]],
        [test_metrics["fn"], test_metrics["tp"]]
    ])

    print(test_cm)

except NameError:
    pass


[STEP 1 - FORWARD]
현재 선택 변수 수: 0
추가 후보 변수 수: 28
추가 변수: NASDAQ_return(%) | gmean=0.6392 | threshold=0.5 | C=1.0 | class_weight=balanced

[STEP 2 - FORWARD]
현재 선택 변수 수: 1
추가 후보 변수 수: 27
추가 변수: KOSPI 200_BB_width | gmean=0.6595 | threshold=0.45 | C=5.0 | class_weight=balanced

[STEP 2 - BACKWARD]
현재 선택 변수 수: 2
선택 변수 중 하나씩 제거하면서 평가 중...
제거 개선폭 -0.020300 <= min_delta 0.001

[STEP 3 - FORWARD]
현재 선택 변수 수: 2
추가 후보 변수 수: 26
추가 변수: KOSPI 200_PPO_Hist | gmean=0.6677 | threshold=0.45 | C=1.0 | class_weight=balanced

[STEP 3 - BACKWARD]
현재 선택 변수 수: 3
선택 변수 중 하나씩 제거하면서 평가 중...
제거 개선폭 -0.008215 <= min_delta 0.001

[STEP 4 - FORWARD]
현재 선택 변수 수: 3
추가 후보 변수 수: 25
추가 변수: KOSPI 200_OG | gmean=0.6771 | threshold=0.45 | C=10.0 | class_weight=balanced

[STEP 4 - BACKWARD]
현재 선택 변수 수: 4
선택 변수 중 하나씩 제거하면서 평가 중...
제거 개선폭 -0.009309 <= min_delta 0.001

[STEP 5 - FORWARD]
현재 선택 변수 수: 4
추가 후보 변수 수: 24
추가 변수: KODEX 200_Premium | gmean=0.6859 | threshold=0.45 | C=5.0 | class_weight=balanced

[STEP 5 - BACKWARD]
현재

,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,0.688329,0.45,0.660714,0.360856,0.239837,0.728395,0.650467,0.689431,0.261416,0.724311,0.348547,696,374,44,118



Validation Confusion Matrix
row=true, col=pred
[[TN, FP],
 [FN, TP]]
[[696 374]
 [ 44 118]]

Step History


,step,action,feature,n_features,gmean,improvement,threshold,acc,f1,precision,recall,spec,balanced_acc,tn,fp,fn,tp,C,class_weight,solver
0,1,add,NASDAQ_return(%),1,0.639233,NaN,0.50,0.711851,0.336449,0.241287,0.555556,0.735514,0.645535,787,283,72,90,1.0,balanced,liblinear
1,2,add,KOSPI 200_BB_width,2,0.659533,0.020300,0.45,0.621753,0.332378,0.216418,0.716049,0.607477,0.661763,650,420,46,116,5.0,balanced,liblinear
2,3,add,KOSPI 200_PPO_Hist,3,0.667747,0.008215,0.45,0.627435,0.339568,0.221388,0.728395,0.612150,0.670272,655,415,44,118,1.0,balanced,liblinear
3,4,add,KOSPI 200_OG,4,0.677056,0.009309,0.45,0.650162,0.349925,0.231537,0.716049,0.640187,0.678118,685,385,46,116,10.0,balanced,liblinear
4,5,add,KODEX 200_Premium,5,0.685899,0.008842,0.45,0.660714,0.358896,0.238776,0.722222,0.651402,0.686812,697,373,45,117,5.0,balanced,liblinear
5,6,add,Signal2_Buy,6,0.688329,0.002431,0.45,0.660714,0.360856,0.239837,0.728395,0.650467,0.689431,696,374,44,118,5.0,balanced,liblinear



Test Metrics


,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,0.656484,0.45,0.607056,0.293217,0.183562,0.728261,0.591781,0.660021,0.203075,0.725968,0.274091,432,298,25,67



Test Confusion Matrix
row=true, col=pred
[[TN, FP],
 [FN, TP]]
[[432 298]
 [ 25  67]]


In [13]:
selected_features

['NASDAQ_return(%)',
 'KOSPI 200_BB_width',
 'KOSPI 200_PPO_Hist',
 'KOSPI 200_OG',
 'KODEX 200_Premium',
 'Signal2_Buy']